<a href="https://colab.research.google.com/github/odenas/modern_ml_hws/blob/main/Copy_of_hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework 4 - Transformers

In this homework, you will build all the components of a Transformer LLM, which can load weights from a (slight variantn of) the Llama 3.2 1B model, and perform inference on it.  While you will _not_ be training the model from scratch in this homework (that will happen in the next homework), this model will contain all the elements to run the LLM, including the basic layers (linear, embedding, RMS norm, multihead attention, etc) and the KV cache mechanism to make inference more efficient.

As with the previous homework, you should import _no_ `torch.nn` modules except those included directly below (the Module, ModuleList, Paramter, and Buffer classes).

In [ ]:
### Run this cell to download and install the necessary modules for the homework
!pip install --upgrade --no-deps git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/modernaicourse/hw4/refs/heads/main/hw4_tests.py

import os
import math
import json
import mugrade
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer

from hw4_tests import (test_Linear, submit_Linear,
                       test_Embedding, submit_Embedding,
                       test_silu, submit_silu,
                       test_RMSNorm, submit_RMSNorm,
                       test_self_attention, submit_self_attention,
                       test_MultiHeadAttention, submit_MultiHeadAttention,
                       test_MultiHeadAttentionKVCache, submit_MultiHeadAttentionKVCache,
                       test_GatedMLP, submit_GatedMLP,
                       test_TransformerBlock, submit_TransformerBlock,
                       test_Llama3Simplified, submit_Llama3Simplified,
                       test_eval_llama3, submit_eval_llama3,
                       test_generate, submit_generate)

os.environ["MUGRADE_HW"] = "Homework 4"
os.environ["MUGRADE_KEY"] = "" ### Your key here

### Question 1 - Linear Layer

Implement a Linear layer, in the same way you did in homework 3.  The one difference here, is that there is no need to explicitly perform an initialization of the layer, i.e., you can use `torch.empty()` to create the weights rather than `torch.randn()` as you did in the previous homework.  This is because you will be loading the weights from the Llama 3.2 model, and thus you don't need to initialize them (it won't hurt anything to initialize them, but it will make creating the model take a bit more time, since these models are starting to get pretty large).

Be sure to store the weights in a `Parameter` called `weight` in the class, or the tests will not pass.

In [ ]:
@mugrade.local_tests
class Linear(Module):
    """ Linear layer with no bias term.  The parameters of the layer are stored in a .weight Parameter"""
    def __init__(self, in_dim, out_dim):
        """
        Initialize a linear layer without a bias term.

        Inputs:
            in_dim : int - input feature dimension
            out_dim : int - output feature dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply the linear layer to one or more input vectors.

        Input:
            X : torch.Tensor[float] (... x in_dim) - input tensor
        Output:
            torch.Tensor[float] (... x out_dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 2 - Embedding layer

As the next step, you will implement an embedding layer.  This is the layer that converts the one-hot encoding of the input into a matrix of input embeddings, e.g., by effectively computing the linear operation $X_{\text{one-hot}} W_E$ where $W_E \in \mathbb{R}^{V \times d}$ is a matrix of embeddings for each of the $V$ tokens (for a few minor technical reasons we won't get in to, it's common to represent $W_E$ without the transpose for embedding layers, e.g., this is what PyTorch does).  

However, note that the input to this layer is not an actual one-hot embedding matrix (it would be very inefficient to store this very sparse matrix explicitly).  Instead, the input is an integer tensor $Y$ that stores the _index_ of each non-zero entry in the one hot embedding.  That is, if the first row of $X_{\text{one-hot}}$ would contain a one in position $i$ and zeros everywhere else, than the first element of $Y$ would be just the integer $i$ (i.e., the elements of $Y$ can range from 0 to `num_tokens-1`).  $Y$ can have any size, though in the common usage of the function, the dimensions would be `(batch_size, sequence_length)` and would return a tensor of size `(batch_size, sequence_length, dim)`.

You should represent the weights with a parameter `.weight` of size `(num_tokens, dim)`.  As with the linear layer, you can initialize this with `torch.empty()` for simplicity.

In [ ]:
@mugrade.local_tests
class Embedding(Module):
    def __init__(self, num_tokens, dim):
        """
        Initialize an embedding table over a fixed vocabulary.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - embedding dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, Y):
        """
        Look up embeddings for an integer tensor of token ids.

        Input:
            Y : torch.Tensor[int] (...) - tensor of token indices in [0, num_tokens)
        Output:
            torch.Tensor[float] (... x dim) - embedding vectors for each token id
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 3 - SiLU nonlinearity

Implement the silu nonlinearity defined as
$$\mathrm{silu}(x) = x \cdot \mathrm{sigmoid}(x)$$

In [ ]:
@mugrade.local_tests
def silu(x):
    """
    Apply the SiLU nonlinearity elementwise.

    Input:
        x : torch.Tensor[float] (...) - input tensor
    Output:
        torch.Tensor[float] (...) - tensor after applying SiLU
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

### Question 4 - RMS Norm

Implement a normalization layer, sometimes referred to as "RMSNorm", which computes the following function:
$$ \mathrm{RMSNorm}(x) = w \circ \frac{x}{\sqrt{\|x\|^2 + \epsilon}} $$
where $\circ$ represents elementwise multiplication, and $w$ is a weight parameters (of size `dim`), which should be initialized to all ones.  Note that it is fairly common for normalization layers to have built-in scaling parameters like this: although in many cases the scaling of the weights could be folded into a subsequent linear layer, for the purposes of training the network it is common to have an explicit set of weights like this (and this is, e.g., what the Llama3 network does).  Be sure to store the weight in a `.weight` Parameter, so that the tests pass correctly.

If applied to a tensor, `RMSNorm` should always be applied along the _last_ dimension.

In [ ]:
@mugrade.local_tests
class RMSNorm(Module):
    def __init__(self, dim, eps=1e-5):
        """
        Initialize an RMS normalization layer over the last dimension.

        Inputs:
            dim : int - size of the final dimension to normalize
            eps : float - numerical stability constant
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply RMS normalization along the final dimension of the input.

        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) - normalized tensor
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 5 - Masked Self Attention

Implement a (masked) self attention operation:
$$Y = \mathrm{softmax} \left ( \frac{ Q K^T}{\sqrt{d}} + \mathrm{M} \right ) V$$
where (in mathematical notation) $Q,K,V \in \mathbb{R}^{T \times d}$ and $M \in \mathbb{R}^{T \times T}$ and where the softmax to the last dimension.  This should work both for the native case above where $Q,K,V$ are all matrices (2D tensors), _and_ for the case where there are additional leading dimensions in the tensors, such as a batch dimension or an additional head dimension for multi-head attention.

In [ ]:
@mugrade.local_tests
def self_attention(Q,K,V, mask=None):
    """
    Apply scaled dot-product attention, optionally with an additive mask.

    Inputs:
        Q : torch.Tensor[float] (... x query_len x d) - query tensor
        K : torch.Tensor[float] (... x key_len x d) - key tensor
        V : torch.Tensor[float] (... x key_len x d_v) - value tensor
        mask : torch.Tensor[float] (... x query_len x key_len) or None - additive attention mask
    Output:
        torch.Tensor[float] (... x query_len x d_v) - attention output tensor
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


### Question 6 - Multi-head Attention (no caching)

We're now going to implement multihead attention.  We'll do this in two parts, first implementing a version without a KV cache (which would be suitable for training an LLM, though we don't do that in this assignment), and later (in the next question) implement a version with a KV cache.

To start, implement the following `MultiHeadAttention` class.  The forward pass of the model should do the followin:
1. Form the $Q,K,V$ tensors by applying the Linear modules `wq`, `wk`, and `wv` (make sure you clear these within the class as element of the `Linear` module you created above).
2. Split $Q,K,V$ along the last dimension to create `n_heads` different heads.
   $$
   Q = \left [\begin{array}{cccc} Q_1 & Q_2 & \cdots & Q_h \end{array} \right ], \;\; K = \left [\begin{array}{cccc} K_1 & K_2 & \cdots & K_h \end{array} \right ], \;\; V = \left [\begin{array}{cccc} V_1 & V_2 & \cdots & V_h \end{array} \right ]$$
   where each $Q_i,K_i,V_i$ would be a sub-block of trailing dimension `dim / n_heads`.

3. Apply the self attention operation to each head
   $$Y = \left [ \begin{array}{cccc} \text{SelfAttn}(Q_1,K_1,V_1)  & \text{SelfAttn}(Q_2,K_2,V_2) & \cdots & \text{SelfAttn}(Q_h,K_h,V_h) \end{array} \right ]$$
4. Apply the final projection operator `wp` to $Y$ (make sure `wp` is also a `Linear` module within the class).

You are welcome to implement this _either_ by explicitly splitting the tensor along the head dimension and using a for loop, _or_ by using tensor reshaping (discussed briefly in class) to make this operation more efficient

In [ ]:
@mugrade.local_tests
class MultiHeadAttention(Module):
    def __init__(self, dim, n_heads):
        """
        Initialize a multi-head self-attention layer without caching.

        Inputs:
            dim : int - total embedding dimension
            n_heads : int - number of attention heads
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention to a batch of sequences.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x seq_len) or None - additive attention mask
            seq_pos : int - starting sequence position (unused here)
            use_kv_cache : bool - whether to use KV caching (unused here)
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

###

### Question 6 - Multi-head Attention (KVCache)

Now implement a version of multihead attention with a KV Cache.  To do this, you should initialize two additional elements in a class, a `.k_cache` and `.v_cache`, both tensors wrapped with the `Buffer` object imported above from PyTorch (this ensures they are converted to the proper datatype when you convert the class, or even put it on a GPU).  These tensors should both be of size `(1,max_cache_size,dim)`, which captures the fact that we are only doing caching for `batch_size=1` for now, as this is commonly how generation is done (though there is no fundamental requirement for this).

In the forward pass, when `use_kv_cache` is True, you need to populate the `seq_pos:seq_pos+X.shape[1]` elements of these caches (along the sequence length dimension, which will be the second dimension) with the `K` and `V` terms you create initially from the input `X`.  Then set the `K` and `V` for use with the `self_attention()` function to be the entire cache up to the `seq_pos+X.shape[1]` position.

Note: updating the cache like this, rather than literally keeping a fixed cache and concatenating it, allows you to re-use the same cache across multiple generations, without having to reset it.

Other than these modifications, the implementation should be exactly the same as in the previous question.


In [ ]:
@mugrade.local_tests
class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim, n_heads, max_cache_size):
        """
        Initialize a multi-head self-attention layer with KV cache buffers.

        Inputs:
            dim : int - total embedding dimension
            n_heads : int - number of attention heads
            max_cache_size : int - maximum sequence length stored in the cache
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 7 - Gated MLP

The Llama3 class of models uses a "gated" MLP rather than a normal two layer network.  It compute the function
$$ \mathrm{GatedMLP}(X) = (\mathrm{silu}(XW_1^T) \circ X W_3^T)W_2^T$$
where $W_1,W_3 \in \mathbb{R}^{d_{\mathrm{ffn} \times d}}$ and $W_2\in \mathbb{R}^{d \times d_{\mathrm{ffn}}}$ and where $\circ$ denotes elementwise multiplication.

Implement this MLP as a Module, being sure to represent the weights as `Linear` layers with the names `w1`, `w2`, `w3`.

In [ ]:
@mugrade.local_tests
class GatedMLP(Module):
    def __init__(self, dim, ffn_dim):
        """
        Initialize the gated feed-forward network used in the transformer block.

        Inputs:
            dim : int - model dimension
            ffn_dim : int - hidden feed-forward dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply the gated feed-forward network to the input tensor.

        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 8 - Transformer Block

Implement a transformer block, defined as the operation
$$\begin{align}
Z &= X + \mathrm{MHA}(\mathrm{norm}_1(X)) \\
Y &= Z + \mathrm{GatedMLP}(\mathrm{norm}_2(Z))
\end{align} $$
Where MHA denotes multihead attention (with a KV Cache), and where we use the notation $\mathrm{RMSNorm}_{1,2}$ to indicate that these are two _different_ RMSNorm layers (i.e., with their own weights).

The multihead attention, norms, and MLP should be stored as the appropriate layer types in the class, with the names `attn`, `norm1`, `norm2`, and `mlp`.




In [ ]:
@mugrade.local_tests
class TransformerBlock(Module):
    def __init__(self, dim, n_heads, ffn_dim, max_cache_size):
        """
        Initialize a transformer block with attention, normalization, and gated MLP layers.

        Inputs:
            dim : int - model dimension
            n_heads : int - number of attention heads
            ffn_dim : int - hidden feed-forward dimension
            max_cache_size : int - maximum sequence length stored in the attention cache
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply one transformer block with residual connections.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - transformed sequence embeddings
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Question 9 - Llama3 Model

Let's finally implement the full Llama3.2 1B model (a slight variant built for this class to fit the network format above, but virtually the same as the original Llama 3.2 1B model).  The model needs to contain the following elements
- `.embedding`: an `Embedding` layer that maps integers of max value `num_tokens` to the embedding dimension `dim`
- `.pos_embeddings`: a `(max_seq_len, dim)` PyTorch `Parameter` (initialized to be empty) that will store the global positional embeddings.
- `.layers`: a PyTorch `ModuleList` of length `num_layers`, where each element is a `TransformerBlock`.
- `.norm`: an `RMSNorm` layer to apply before the output layer
- `.output`: a `Linear` layer that converts `dim` dimensional inputs to `num_tokens` dimensional outputs.
- `.mask`: a PyTorch `Buffer` containing a `(max_seq_len, max_seq_len)` strictly upper triangular mask (negative infinity in strictly upper diagonal entries, zero elsewhere).  You will pass subsets of this mask to the transformer layers based upon the actual sequence position of the token inputs.

The full set of operations performed by the model in the forward pass should be the following:
1. Apply the `.embedding` layer to the input `tokens`, and add `.pos_embeddings` for the appropriate sequence position.
2. Apply each element of `.layers` sequentially, using the appropriate subset of mask.
3. Return `.output` applied to `.norm` of the output of the last layer.

We also include (you don't have to write this, but it could help to verify that your objects are all named correctly), code to load the data from a `consolidated.00.pth` file, which is the file format for the model that you'll download in the next portion.

In [ ]:
@mugrade.local_tests
class Llama3Simplified(Module):
    def __init__(self, num_tokens, dim, n_heads, max_seq_len, ffn_dim, num_layers):
        """
        Initialize the simplified Llama 3 model used in this homework.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - model dimension
            n_heads : int - number of attention heads per layer
            max_seq_len : int - maximum supported sequence length
            ffn_dim : int - hidden feed-forward dimension in each block
            num_layers : int - number of transformer blocks
        """
        super().__init__()
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def forward(self, tokens, seq_pos=0, use_kv_cache=False):
        """
        Apply the full language model to a batch of token sequences.

        Inputs:
            tokens : torch.Tensor[int] (batch_size x seq_len) - input token ids
            seq_pos : int - starting sequence position for positional embeddings and cached attention
            use_kv_cache : bool - whether to update and use cached keys and values
        Output:
            torch.Tensor[float] (batch_size x seq_len x num_tokens) - output logits
        """
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def load_llama_weights(self, checkpoint):
        self.embedding.weight.data = checkpoint["tok_embeddings.weight"]
        self.pos_embeddings.data = checkpoint["pos_embeddings.weight"]
        self.norm.weight.data = checkpoint["norm.weight"]
        self.output.weight.data = checkpoint["output.weight"]

        for i,layer in enumerate(self.layers):
            layer.attn.wq.weight.data = checkpoint[f"layers.{i}.attention.wq.weight"]
            layer.attn.wk.weight.data = checkpoint[f"layers.{i}.attention.wk.weight"]
            layer.attn.wv.weight.data = checkpoint[f"layers.{i}.attention.wv.weight"]
            layer.attn.wp.weight.data = checkpoint[f"layers.{i}.attention.wo.weight"]

            layer.mlp.w1.weight.data = checkpoint[f"layers.{i}.feed_forward.w1.weight"]
            layer.mlp.w2.weight.data = checkpoint[f"layers.{i}.feed_forward.w2.weight"]
            layer.mlp.w3.weight.data = checkpoint[f"layers.{i}.feed_forward.w3.weight"]

            layer.norm1.weight.data = checkpoint[f"layers.{i}.attention_norm.weight"]
            layer.norm2.weight.data = checkpoint[f"layers.{i}.ffn_norm.weight"]


If you implemented the above correctly, you should be able to download the model configuration and weights using the following command.

In [ ]:
from huggingface_hub import hf_hub_download

repo = "zkolter/Llama-3.2-1B-Instruct-Simplified"
for filename in ["consolidated.00.pth", "params.json", "tokenizer.model", "tokenizer.py",]:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="model", local_dir=".")

checkpoint = torch.load("consolidated.00.pth", map_location=torch.device('cpu'))
with open("params.json", "rt") as f:
    params = json.load(f)

model = Llama3Simplified(params['vocab_size'],
                         params['dim'],
                         params['n_heads'],
                         params['max_seq_len'],
                         params['dim'] * params['ffn_dim_multiplier'],
                         params['n_layers'])
model.load_llama_weights(checkpoint)
model = model.float()


@mugrade.local_tests
def eval_llama3():
    return model


### Question 10 - Generation

Last, write a function that will sequentially generate a response from the LLM sequentially given a prompt.  The process should work as follows:
1. First, the model on the full prompt to predict the next output token. Then sample from the softmax distribution applied to this distribution, scaled by dividing by `temp`.
2. Repeatedly append this token the sequence, re-run the model (using KV cache) to predict the distribution over next tokens and sample from the temperature-scaled softmax distribution.
3. If `verbose` is True, after each generated token, print it's string representation, using the call `tokenizer.decode(token)`.  Additionally, check if each generated token is in the set `tokenizer.stop_tokens`, and break out of the generation loop if so (even if the system has not generated `max_tokens` tokens).



In [ ]:
@mugrade.local_tests
def generate(model, prompt_tokens, tokenizer, temp=0.7, max_tokens = 500, verbose=True):
    """
    Autoregressively sample tokens from a language model using its KV cache.

    Inputs:
        model : Module - language model mapping token sequences to logits
        prompt_tokens : list[int] - initial prompt tokens
        tokenizer : object - tokenizer with decode() and stop_tokens
        temp : float - sampling temperature
        max_tokens : int - maximum number of new tokens to generate
        verbose : bool - whether to print each generated token as it is sampled
    Output:
        list[int] - generated tokens, excluding the prompt tokens
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


If you have implemented this all correctly, the following code should generate a response to the question "Why is the sky blue?"  Note the details of the code here are not that critical, this is just using the official Llama 3 tokenizer and chat formatting to put the prompt into the canonical Llama 3 chat format.

In [ ]:
from tokenizer import Tokenizer, Message, ChatFormat
tokenizer = Tokenizer("tokenizer.model")
chat = ChatFormat(tokenizer)
msg = Message(role="user", content="Why is the sky blue?")
prompt = chat.encode_dialog_prompt([msg])

generate(model, prompt, tokenizer)